In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-U", "huggingface_hub"],
    check=True,
)

In [ ]:
BASE = Path("/content/drive/MyDrive/AIMOProofPilot Container")
TARGET = BASE / "models" / "judge"
CACHE = BASE / ".hf_cache" / "judge"
TEMP = BASE / ".tmp" / "judge"
REPO_ID = "openai/gpt-oss-120b"
INCLUDE_PATTERNS = [
    ".gitattributes",
    "LICENSE",
    "README.md",
    "USAGE_POLICY",
    "*.json",
    "*.jinja",
    "*.safetensors",
]
EXCLUDE_PATTERNS = [
    ".cache/**",
    "metal/**",
    "original/**",
]

TARGET.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
TEMP.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(CACHE)
os.environ["HF_HUB_CACHE"] = str(CACHE / "hub")
os.environ["TMPDIR"] = str(TEMP)
os.environ["TEMP"] = str(TEMP)
os.environ["TMP"] = str(TEMP)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "0"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    local_dir=str(TARGET),
    cache_dir=str(CACHE),
    allow_patterns=INCLUDE_PATTERNS,
    ignore_patterns=EXCLUDE_PATTERNS,
    max_workers=1,
)

In [ ]:
def model_files(root: Path) -> list[Path]:

    return [
        path
        for path in root.rglob("*")
        if path.is_file() and ".cache" not in path.parts
    ]


required_files = [
    "LICENSE",
    "README.md",
    "USAGE_POLICY",
    "chat_template.jinja",
    "config.json",
    "generation_config.json",
    "model.safetensors.index.json",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
]
missing_files = [
    name
    for name in required_files
    if not (TARGET / name).exists()
]

if missing_files:
    raise FileNotFoundError(f"Missing required files: {missing_files}")

safetensor_files = sorted(TARGET.glob("*.safetensors"))

if not safetensor_files:
    raise FileNotFoundError("No safetensors shards were downloaded.")

total_bytes = sum(path.stat().st_size for path in model_files(TARGET))
print(f"Directory: {TARGET}")
print(f"Model files: {len(model_files(TARGET))}")
print(f"Safetensors shards: {len(safetensor_files)}")
print(f"Total model bytes: {total_bytes:,}")

for path in safetensor_files:
    print(f"{path.name} {path.stat().st_size:,}")